# Physics-Informed Digital Twin Framework
## 현장 레퍼런스용 범용 PINN 통합 파이프라인

### 📌 문서 목적
- **기존 프로젝트 확장**: Feature Engineering → Surrogate Model → Bayesian Optimization 파이프라인에 PINN 통합
- **현장 범용성**: 공정/메커니즘이 달라도 PhysicsRegistry만 교체하면 재사용 가능
- **디지털 트윈 지향**: 실시간 센서 데이터 → 물리 법칙 검증 → 최적 제어 출력

### 🏗️ 전체 아키텍처
```
[센서 데이터] → [Physics-Aware FE] → [Hybrid PINN Surrogate] → [Bayesian Opt] → [Digital Twin]
                        ↑                      ↑
                 PhysicsRegistry         PINN Physics Loss
                 (공정별 교체)          λ × residual penalty
```

### ⚡ 3가지 운영 모드 (속도 ↔ 정확도 트레이드오프)
| Mode | 구성 | 추론속도 | 물리 정합성 | 권장 상황 |
|------|------|---------|------------|---------|
| **A: Fast** | XGBoost + Physics Features | < 1ms | 간접 | 실시간 제어, 엣지 디바이스 |
| **B: Balanced** | XGBoost 추론 + PINN 검증 | ~5ms | 직접 검증 | **현장 기본값 (권장)** |
| **C: Accurate** | Pure PINN | ~50ms | 완전 통합 | 설계 최적화, 배치 분석 |


# 1. 기본 환경 세팅 (기존 코드 보존)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── 기존 ML 라이브러리 ────────────────────────────────────────
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import xgboost as xgb
from xgboost import XGBRegressor

# ── PINN 추가 라이브러리 ─────────────────────────────────────
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
    print(f"✅ PyTorch {torch.__version__} 사용 가능")
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"   Device: {DEVICE}")
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠️  PyTorch 미설치. Mode A만 사용 가능합니다.")
    print("   설치: pip install torch")

# ── 시각화 설정 ──────────────────────────────────────────────
plt.rcParams['axes.unicode_minus'] = False
try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
except:
    pass

from IPython.display import display, HTML
display(HTML("<style>.container{width:100% !important;}</style>"))
pd.set_option('display.max_columns', 100)

✅ PyTorch 2.10.0+cpu 사용 가능
   Device: cpu


# 2. PhysicsRegistry — 범용 물리 법칙 플러그인 시스템

## 🔧 설계 원칙
- **PhysicsLaw**: 단일 물리 법칙의 인터페이스 (잔차 계산)
- **PhysicsRegistry**: 여러 법칙을 묶어 총합 물리 손실(Physics Loss) 생성
- **공정 교체 방법**: PhysicsRegistry에 해당 공정의 Law만 등록하면 됨

### 현재 구현 법칙 (전력 계통 기준)
| 법칙 | 수식 | 물리적 의미 |
|------|------|------------|
| PowerTriangle | S² = P² + Q² | 전력 삼각형 보존 |
| EnergyConservation | ΔP ≥ 0 (급격한 변화 억제) | 에너지 연속성 |
| PowerFactor | PF = P/S ∈ [0,1] | 역률 물리 범위 |
| KEPCOCapacitorStrategy | 피크 시간대 진상 역률 발생 방지 | 한전 월평균 PF 전략 허용 + 진상 역률만 제약 |


In [2]:
import torch
import torch.nn as nn
from abc import ABC, abstractmethod
from typing import Dict, List, Optional
import numpy as np

# ══════════════════════════════════════════════════════════════
# [PINN Block 1] 추상 인터페이스: PhysicsLaw
# ══════════════════════════════════════════════════════════════
class PhysicsLaw(ABC):
    """
    단일 물리 법칙의 추상 기반 클래스.
    
    새로운 공정에 적용 시:
    1. 이 클래스를 상속받아 구현
    2. residual() 메서드에서 '이상적으로 0이어야 할 값' 반환
    3. PhysicsRegistry에 등록
    """
    def __init__(self, name: str, weight: float = 1.0):
        self.name = name
        self.weight = weight  # 다른 법칙 대비 상대적 중요도
    
    @abstractmethod
    def residual(self, inputs: Dict[str, torch.Tensor], 
                 outputs: Dict[str, torch.Tensor]) -> torch.Tensor:
        """
        물리 법칙의 잔차(residual)를 반환.
        완벽한 물리 정합 시 → 0
        위반 시 → 0 이상의 값
        """
        pass
    
    def loss(self, inputs, outputs) -> torch.Tensor:
        res = self.residual(inputs, outputs)
        return self.weight * torch.mean(res ** 2)


# ══════════════════════════════════════════════════════════════
# [PINN Block 2] 전력 계통용 물리 법칙 구현체
# ══════════════════════════════════════════════════════════════

class PowerTriangleLaw(PhysicsLaw):
    """
    전력 삼각형 보존 법칙: S² = P² + Q_net²
    
    잔차: |S_measured - sqrt(P² + Q_net²)| / S_measured
    - P: 유효전력 (Usage_kWh)
    - Q_lag: 지상 무효전력 (Lagging_Current_Reactive_Power_kVarh)
    - Q_lead: 진상 무효전력 (Leading_Current_Reactive_Power_kVarh)
    - S: 피상전력 (Apparent_Power)
    """
    def __init__(self, weight: float = 1.0):
        super().__init__('PowerTriangle', weight)
    
    def residual(self, inputs, outputs):
        P = outputs.get('P', inputs.get('P'))
        Q_lag = inputs.get('Q_lag', torch.zeros_like(P))
        Q_lead = inputs.get('Q_lead', torch.zeros_like(P))
        S_measured = inputs.get('S', None)
        
        Q_net = Q_lag - Q_lead
        S_calc = torch.sqrt(P**2 + Q_net**2 + 1e-8)
        
        if S_measured is not None:
            # 실측 피상전력과 계산값의 차이
            return (S_calc - S_measured) / (S_measured + 1e-8)
        else:
            # 피상전력 없을 때: 비율 일관성만 검증
            return torch.zeros_like(P)


class PowerFactorLaw(PhysicsLaw):
    """
    역률 물리 범위 법칙: PF ∈ [0, 1]
    
    잔차: max(0, PF - 1) + max(0, -PF)  → PF가 [0,1] 밖이면 패널티
    """
    def __init__(self, weight: float = 0.5):
        super().__init__('PowerFactor', weight)
    
    def residual(self, inputs, outputs):
        P = outputs.get('P', inputs.get('P'))
        S = inputs.get('S', None)
        
        if S is None:
            Q_lag = inputs.get('Q_lag', torch.zeros_like(P))
            Q_lead = inputs.get('Q_lead', torch.zeros_like(P))
            Q_net = Q_lag - Q_lead
            S = torch.sqrt(P**2 + Q_net**2 + 1e-8)
        
        PF = P / (S + 1e-8)
        # PF는 반드시 [0, 1] 범위 내
        upper_violation = torch.relu(PF - 1.0)
        lower_violation = torch.relu(-PF)
        return upper_violation + lower_violation


class EnergySmoothnessLaw(PhysicsLaw):
    """
    에너지 연속성 법칙: 전력의 급격한 변동 억제
    
    물리적 근거: 관성이 있는 시스템에서 전력이 순간적으로
    극단적으로 변할 수 없음 (모터, 변압기 등)
    
    잔차: max(0, |ΔP| - threshold) → 임계값 초과 변동 패널티
    """
    def __init__(self, weight: float = 0.3, threshold: float = 0.5):
        super().__init__('EnergySmoothing', weight)
        self.threshold = threshold  # 정규화된 스케일 기준 허용 변동폭
    
    def residual(self, inputs, outputs):
        P = outputs.get('P', inputs.get('P'))
        if len(P) < 2:
            return torch.zeros(1)
        delta_P = torch.abs(P[1:] - P[:-1])
        violation = torch.relu(delta_P - self.threshold)
        # 길이 맞춤
        return torch.cat([torch.zeros(1, device=P.device), violation])


class KEPCOCapacitorStrategyLaw(PhysicsLaw):
    """
    한전 역률 패널티 구조를 반영한 콘덴서 전략 법칙.

    ──────────────────────────────────────────────────────────
    [기존 CapacitorBalanceLaw 설계 오류 및 수정 근거]

    기존 구현의 가정:
        "Motor_Rate < 5% & Capacitor_Rate > 60% → 과보상 위반"

    실제 문제:
        한전 역률 페널티는 '월간 누적 무효전력량 평균'으로 산정된다.
        따라서 경부하 시간대(야간·주말)에 모터가 멈춰 있더라도
        콘덴서를 의도적으로 고가동해 월평균 PF를 끌어올리는 것은
        현장에서 실제로 사용하는 합리적 비용 최적화 전략이다.
        실제 데이터에서도 이 패턴이 관찰된 바 있다.

        → 기존 법칙이 이 정상 운영을 '물리 위반'으로 잘못 패널티 부과했음.
        → PINN이 이 합리적 전략을 억제하면 오히려 비용 최적화를 방해함.

    수정 방향 — 2가지 구간 명시적 분리:
        ① 전략적 보상 구간 (경부하·비피크):
           - 모터 정지 상태이더라도 비피크 시간대 콘덴서 고가동
           → 잔차 = 0 (허용). PINN이 이 전략을 학습하도록 방치.

        ② 물리적 비효율 구간 (피크 시간대 진상 역률 발생):
           - 피크 시간대에 Q_lead > Q_lag → 합성 무효전력이 진상으로 반전
           → 한전 진상 역률 페널티 발생 + 설비 전압 상승 위험
           → 이 구간만 패널티 부여 (실제 비효율)

    한전 역률 페널티 기준 (참고):
        지상 역률 관리: 09:00~23:00, 기준 PF 0.90
        진상 역률 관리: 23:00~09:00, 기준 PF 0.95
        (둘 다 위반 시 기본요금에 위반 % 가산)
    ──────────────────────────────────────────────────────────

    Args:
        peak_start:       지상역률 페널티 시작 시각 (시, 기본 9)
        peak_end:         지상역률 페널티 종료 시각 (시, 기본 23)
        weight:           다른 법칙 대비 상대 가중치
    """
    def __init__(
        self,
        weight: float = 0.4,
        peak_start: float = 9.0,
        peak_end: float = 23.0,
    ):
        super().__init__('KEPCOCapacitorStrategy', weight)
        self.peak_start = peak_start
        self.peak_end   = peak_end

    def residual(self, inputs, outputs):
        """
        잔차 = (피크 시간대 가중치) × relu(Q_lead - Q_lag)

        경부하·비피크 구간의 전략적 콘덴서 고가동:
            is_peak ≈ 0 → 잔차 ≈ 0 (허용)

        피크 시간대 진상 역률 발생:
            is_peak ≈ 1, Q_lead > Q_lag → 잔차 > 0 (패널티)
        """
        hour   = inputs.get('hour',  torch.zeros(1))
        Q_lag  = inputs.get('Q_lag', torch.zeros(1))
        Q_lead = inputs.get('Q_lead',torch.zeros(1))
        P      = outputs.get('P', inputs.get('P', torch.ones(1)))

        # 피크 시간대 소프트 지시자 (sigmoid로 경계 부드럽게)
        is_after_start = torch.sigmoid(2.0 * (hour - self.peak_start))
        is_before_end  = torch.sigmoid(2.0 * (self.peak_end - hour))
        is_peak = is_after_start * is_before_end  # shape: 배치 크기

        # 진상 역률 발생량: Q_lead > Q_lag 이면 양수
        leading_excess = torch.relu(Q_lead - Q_lag)

        # 피크 시간대 진상 역률만 패널티, 정규화
        return is_peak * leading_excess / (torch.abs(P) + 1e-8)
# ══════════════════════════════════════════════════════════════
# [PINN Block 3] PhysicsRegistry: 법칙 묶음 관리자
# ══════════════════════════════════════════════════════════════

class PhysicsRegistry:
    """
    물리 법칙들의 레지스트리.
    
    공정 교체 방법:
        registry = PhysicsRegistry()
        registry.register(MyCustomLaw(weight=1.0))
        # → 동일한 PINN 파이프라인에서 새 공정 자동 적용
    
    사전 정의 팩토리:
        PhysicsRegistry.power_system()  # 전력 계통 (현재 프로젝트)
        PhysicsRegistry.chemical()      # 화학 공정 (예시)
        PhysicsRegistry.general()       # 범용 (법칙 없음)
    """
    def __init__(self, name: str = "custom"):
        self.name = name
        self._laws: List[PhysicsLaw] = []
    
    def register(self, law: PhysicsLaw):
        self._laws.append(law)
        print(f"  ✅ 등록: {law.name} (weight={law.weight})")
        return self
    
    def total_physics_loss(self, inputs: Dict, outputs: Dict) -> torch.Tensor:
        """전체 물리 손실 = Σ(각 법칙의 가중 손실)"""
        if not self._laws:
            return torch.tensor(0.0)
        losses = [law.loss(inputs, outputs) for law in self._laws]
        return torch.stack(losses).sum()
    
    def validate_numpy(self, df_row: Dict) -> Dict:
        """numpy 배열로 물리 법칙 위반 여부 점검 (추론 시간용)"""
        report = {}
        for law in self._laws:
            inputs = {k: torch.tensor([v], dtype=torch.float32) 
                      for k, v in df_row.items() if isinstance(v, (int, float))}
            outputs = {'P': inputs.get('P', torch.tensor([0.0]))}
            try:
                res = law.residual(inputs, outputs).item()
                report[law.name] = {'residual': res, 'violation': abs(res) > 0.05}
            except:
                report[law.name] = {'residual': None, 'violation': False}
        return report
    
    @classmethod
    def power_system(cls):
        """전력 계통용 레지스트리 (현재 프로젝트)"""
        print("\n🔌 전력 계통 PhysicsRegistry 초기화 중...")
        reg = cls(name="power_system")
        reg.register(PowerTriangleLaw(weight=1.0))
        reg.register(PowerFactorLaw(weight=0.5))
        reg.register(EnergySmoothnessLaw(weight=0.3))
        reg.register(KEPCOCapacitorStrategyLaw(weight=0.4))
        return reg
    
    @classmethod
    def chemical_process(cls):
        """
        화학 공정 예시 (현장 적용 시 CustomLaw 구현 필요)
        
        구현 예정 법칙:
        - MassBalanceLaw:    Σ(입력 유량) = Σ(출력 유량) + 누적량 변화
        - EnergyBalanceLaw: Q_reaction = ΔH * F (반응 열량)
        - ArrheniusLaw:     k = A * exp(-Ea / RT) (반응 속도)
        """
        print("\n⚗️  화학 공정 PhysicsRegistry (템플릿 - 법칙 구현 필요)")
        reg = cls(name="chemical_process")
        # 여기에 화학 공정 법칙 등록
        # reg.register(MassBalanceLaw(weight=1.0))
        # reg.register(EnergyBalanceLaw(weight=0.8))
        return reg
    
    @classmethod
    def general(cls):
        """범용 레지스트리 (물리 법칙 없음 - 데이터만 학습)"""
        print("\n📊 범용 PhysicsRegistry (데이터 전용 모드)")
        return cls(name="general")


# ── 현재 프로젝트에 맞는 레지스트리 생성 ────────────────────
physics = PhysicsRegistry.power_system()
print(f"\n총 {len(physics._laws)}개 물리 법칙 등록 완료")


🔌 전력 계통 PhysicsRegistry 초기화 중...
  ✅ 등록: PowerTriangle (weight=1.0)
  ✅ 등록: PowerFactor (weight=0.5)
  ✅ 등록: EnergySmoothing (weight=0.3)
  ✅ 등록: KEPCOCapacitorStrategy (weight=0.4)

총 4개 물리 법칙 등록 완료


# 3. PINN 서로게이트 모델 (Physics-Informed Neural Network)

## Mode C: Pure PINN — 높은 정확도, 상대적으로 느린 추론
"""
손실 함수 설계:
  L_total = L_data + λ_physics × L_physics
  
  - L_data    = MSE(y_pred, y_true)           ← 데이터 학습
  - L_physics = Σ PhysicsLaw.loss(inputs, outputs)  ← 물리 제약
  - λ_physics = 0.01 ~ 0.5 (튜닝 하이퍼파라미터)

λ 선택 가이드:
  λ = 0.01: 데이터 우선, 물리는 소프트 가이드
  λ = 0.1 : 균형 (권장)
  λ = 0.5+: 물리 우선, 데이터 부족 시 사용
"""


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from typing import Dict, List, Optional, Tuple
import time

class PINNSurrogate(nn.Module):
    """
    Physics-Informed Neural Network Surrogate Model
    
    설계 원칙:
    1. 네트워크 구조는 범용 MLP (공정 독립적)
    2. 물리 법칙은 PhysicsRegistry를 통해 외부에서 주입
    3. λ_physics로 데이터/물리 학습 비중 조절 가능
    
    Args:
        input_dim: 입력 피처 수
        output_dim: 출력 변수 수 (기본: 1)
        hidden_dims: 은닉층 크기 리스트
        physics_registry: 적용할 물리 법칙 레지스트리
        dropout_rate: 드롭아웃 비율 (과적합 방지)
    """
    def __init__(
        self,
        input_dim: int,
        output_dim: int = 1,
        hidden_dims: List[int] = [128, 64, 32],
        physics_registry: Optional['PhysicsRegistry'] = None,
        dropout_rate: float = 0.1
    ):
        super().__init__()
        self.physics_registry = physics_registry
        self.input_dim = input_dim
        self.output_dim = output_dim
        
        # ── MLP 구성 (Batch Norm + Dropout 포함) ─────────────
        layers = []
        dims = [input_dim] + hidden_dims
        for i in range(len(dims) - 1):
            layers.extend([
                nn.Linear(dims[i], dims[i+1]),
                nn.BatchNorm1d(dims[i+1]),
                nn.GELU(),
                nn.Dropout(dropout_rate)
            ])
        layers.append(nn.Linear(hidden_dims[-1], output_dim))
        self.network = nn.Sequential(*layers)
        
        # 가중치 초기화 (He initialization)
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)
    
    def compute_losses(
        self, 
        x: torch.Tensor, 
        y_true: torch.Tensor,
        physics_inputs: Optional[Dict] = None,
        lambda_physics: float = 0.1
    ) -> Tuple[torch.Tensor, Dict]:
        """
        총 손실 계산: L = L_data + λ × L_physics
        
        Returns:
            total_loss: 역전파에 사용할 최종 손실
            loss_dict: 각 손실 항목 (모니터링용)
        """
        y_pred = self.forward(x)
        
        # 데이터 손실 (MSE)
        l_data = nn.functional.mse_loss(y_pred, y_true)
        
        # 물리 손실
        l_physics = torch.tensor(0.0, device=x.device)
        if self.physics_registry and physics_inputs is not None:
            outputs = {'P': y_pred}
            l_physics = self.physics_registry.total_physics_loss(physics_inputs, outputs)
        
        total = l_data + lambda_physics * l_physics
        
        loss_dict = {
            'total': total.item(),
            'data': l_data.item(),
            'physics': l_physics.item()
        }
        return total, loss_dict
    
    def predict_numpy(self, X: np.ndarray) -> np.ndarray:
        """numpy 배열 입력 → numpy 배열 출력 (추론용)"""
        self.eval()
        with torch.no_grad():
            X_t = torch.FloatTensor(X).to(next(self.parameters()).device)
            return self.forward(X_t).cpu().numpy().flatten()


# ══════════════════════════════════════════════════════════════
# PINN 학습 유틸리티
# ══════════════════════════════════════════════════════════════

class PINNTrainer:
    """
    PINN 학습 루프 관리자.
    
    Early stopping, 스케줄러, 학습 곡선 기록을 포함한
    안정적인 학습 루프 제공.
    """
    def __init__(
        self, 
        model: PINNSurrogate,
        device: str = 'cpu',
        learning_rate: float = 1e-3,
        lambda_physics: float = 0.1
    ):
        self.model = model.to(device)
        self.device = device
        self.lambda_physics = lambda_physics
        self.optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=100)
        self.history = {'total': [], 'data': [], 'physics': [], 'val_mae': []}
    
    def prepare_loaders(
        self, X_train, y_train, X_val, y_val, batch_size=256
    ) -> Tuple[DataLoader, DataLoader]:
        """numpy 배열 → PyTorch DataLoader 변환"""
        to_t = lambda a: torch.FloatTensor(a).to(self.device)
        
        train_ds = TensorDataset(
            to_t(X_train), 
            to_t(y_train.reshape(-1, 1))
        )
        val_ds = TensorDataset(
            to_t(X_val), 
            to_t(y_val.reshape(-1, 1))
        )
        return (
            DataLoader(train_ds, batch_size=batch_size, shuffle=True),
            DataLoader(val_ds, batch_size=batch_size)
        )
    
    def train(
        self, 
        train_loader: DataLoader, 
        val_loader: DataLoader,
        epochs: int = 100,
        patience: int = 15,
        verbose: bool = True
    ) -> Dict:
        """학습 루프 실행"""
        best_val = float('inf')
        patience_cnt = 0
        best_state = None
        
        print(f"\n{'='*60}")
        print(f"PINN 학습 시작 | λ_physics={self.lambda_physics} | device={self.device}")
        print(f"{'='*60}")
        
        for epoch in range(epochs):
            # Train
            self.model.train()
            epoch_losses = {'total': 0, 'data': 0, 'physics': 0}
            
            for X_batch, y_batch in train_loader:
                self.optimizer.zero_grad()
                loss, loss_dict = self.model.compute_losses(
                    X_batch, y_batch, lambda_physics=self.lambda_physics
                )
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
                
                for k in epoch_losses:
                    epoch_losses[k] += loss_dict[k]
            
            # Validation
            self.model.eval()
            val_preds, val_trues = [], []
            with torch.no_grad():
                for X_b, y_b in val_loader:
                    pred = self.model(X_b)
                    val_preds.append(pred.cpu().numpy())
                    val_trues.append(y_b.cpu().numpy())
            
            val_mae = mean_absolute_error(
                np.concatenate(val_trues), np.concatenate(val_preds)
            )
            
            # 기록
            n = len(train_loader)
            for k in epoch_losses:
                self.history[k].append(epoch_losses[k] / n)
            self.history['val_mae'].append(val_mae)
            
            self.scheduler.step()
            
            # Early stopping
            if val_mae < best_val:
                best_val = val_mae
                patience_cnt = 0
                best_state = {k: v.clone() for k, v in self.model.state_dict().items()}
            else:
                patience_cnt += 1
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1:3d} | "
                      f"Loss={self.history['total'][-1]:.4f} "
                      f"(data={self.history['data'][-1]:.4f}, "
                      f"physics={self.history['physics'][-1]:.4f}) | "
                      f"Val MAE={val_mae:.3f}")
            
            if patience_cnt >= patience:
                print(f"\n🛑 Early stopping at epoch {epoch+1} (best Val MAE: {best_val:.4f})")
                break
        
        if best_state:
            self.model.load_state_dict(best_state)
        
        print(f"\n✅ 학습 완료 | Best Val MAE: {best_val:.4f}")
        return self.history
    
    def plot_history(self):
        """학습 곡선 시각화"""
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        axes[0].plot(self.history['total'], label='Total Loss', color='black', lw=2)
        axes[0].plot(self.history['data'], label='Data Loss', color='dodgerblue', lw=1.5)
        axes[0].plot(self.history['physics'], label='Physics Loss', color='tomato', lw=1.5)
        axes[0].set_title('PINN Training Losses')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend()
        axes[0].set_yscale('log')
        axes[0].grid(True, alpha=0.3)
        
        axes[1].plot(self.history['val_mae'], color='green', lw=2)
        axes[1].set_title('Validation MAE')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('MAE (kWh)')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

print("✅ PINNSurrogate 및 PINNTrainer 클래스 정의 완료")

✅ PINNSurrogate 및 PINNTrainer 클래스 정의 완료


# 4. HybridSurrogate — Mode B (XGBoost + PINN 검증층)

## ⚡ 핵심 아이디어
- XGBoost: 빠른 추론 (~1ms), 기존 성능 유지
- PINN: 물리 위반 여부를 비동기로 검증
- 위반 감지 시: PINN 보정값으로 교체하거나 알람 발생

## 속도 벤치마크 (참고값)
| 모델 | 추론 1회 | 1000회 배치 |
|------|---------|-----------|
| XGBoost only | ~0.1ms | ~50ms |
| XGBoost + PINN 검증 | ~3ms | ~200ms |
| Pure PINN | ~20ms | ~800ms |


In [4]:
class HybridSurrogate:
    """
    Mode B: XGBoost 빠른 추론 + PINN 물리 검증 레이어
    
    운영 로직:
    1. 기본 추론: XGBoost (항상 빠름)
    2. 배치 검증: PINN 잔차로 물리 위반 여부 확인
    3. 위반 시: PINN 보정값 사용 또는 알람 발동
    
    Args:
        xgb_model: 기존 XGBoost 서로게이트 모델
        pinn_model: 학습된 PINNSurrogate
        scaler: 피처 스케일러 (numpy → PINN 입력 변환용)
        physics_registry: 물리 검증 레지스트리
        correction_threshold: 물리 잔차 임계값 (이 이상이면 보정)
    """
    def __init__(
        self,
        xgb_model,
        pinn_model: Optional[PINNSurrogate],
        scaler,
        physics_registry: PhysicsRegistry,
        correction_threshold: float = 0.1
    ):
        self.xgb = xgb_model
        self.pinn = pinn_model
        self.scaler = scaler
        self.physics = physics_registry
        self.threshold = correction_threshold
        self.correction_log = []
    
    def predict(
        self, 
        X: np.ndarray, 
        mode: str = 'balanced',
        df_physics: Optional[pd.DataFrame] = None
    ) -> Dict:
        """
        통합 예측 메서드.
        
        Args:
            X: 피처 배열 (스케일링 전)
            mode: 'fast' | 'balanced' | 'accurate'
            df_physics: 물리 검증용 원본 피처 (kVarh 등)
        
        Returns:
            dict with 'prediction', 'source', 'physics_violations'
        """
        result = {
            'prediction': None,
            'source': 'xgb',
            'physics_violations': {},
            'corrected': False
        }
        
        # ── Mode A: Fast (XGBoost only) ─────────────────────
        if mode == 'fast':
            result['prediction'] = self.xgb.predict(X)
            result['source'] = 'xgb_fast'
            return result
        
        # ── Mode C: Accurate (Pure PINN) ────────────────────
        if mode == 'accurate' and self.pinn is not None:
            X_scaled = self.scaler.transform(X)
            result['prediction'] = self.pinn.predict_numpy(X_scaled)
            result['source'] = 'pinn_accurate'
            return result
        
        # ── Mode B: Balanced (XGBoost + PINN 검증) ─────────
        xgb_pred = self.xgb.predict(X)
        result['prediction'] = xgb_pred
        result['source'] = 'xgb_balanced'
        
        # 물리 검증 (PINN 또는 직접 계산)
        if df_physics is not None:
            violations = self._check_physics(xgb_pred, df_physics)
            result['physics_violations'] = violations
            
            # 위반 심각도가 임계값 초과 시 PINN 보정
            if self.pinn is not None and self._has_critical_violation(violations):
                X_scaled = self.scaler.transform(X)
                pinn_pred = self.pinn.predict_numpy(X_scaled)
                result['prediction'] = pinn_pred
                result['source'] = 'pinn_corrected'
                result['corrected'] = True
                self.correction_log.append({
                    'n_corrected': len(X),
                    'violations': violations
                })
        
        return result
    
    def _check_physics(self, predictions: np.ndarray, df: pd.DataFrame) -> Dict:
        """물리 법칙 위반 여부 직접 계산"""
        violations = {}
        
        # 전력 삼각형 검증
        if all(c in df.columns for c in ['Lagging_Current_Reactive_Power_kVarh',
                                          'Leading_Current_Reactive_Power_kVarh',
                                          'Apparent_Power']):
            P = predictions
            Q_net = (df['Lagging_Current_Reactive_Power_kVarh'].values - 
                     df['Leading_Current_Reactive_Power_kVarh'].values)
            S_calc = np.sqrt(P**2 + Q_net**2 + 1e-8)
            S_meas = df['Apparent_Power'].values
            residual = np.abs(S_calc - S_meas) / (S_meas + 1e-8)
            
            violations['PowerTriangle'] = {
                'mean_residual': float(residual.mean()),
                'max_residual': float(residual.max()),
                'violation_rate': float((residual > 0.05).mean()),
                'status': '⚠️ 위반' if residual.mean() > self.threshold else '✅ 정상'
            }
        
        # 역률 범위 검증
        if 'PF_Physical' in df.columns:
            pf = predictions / (np.sqrt(predictions**2 + 
                df.get('Q_total_abs', pd.Series(np.zeros(len(df)))).values**2) + 1e-8)
            range_violation = np.mean((pf < 0) | (pf > 1))
            violations['PowerFactor'] = {
                'out_of_range_rate': float(range_violation),
                'status': '⚠️ 위반' if range_violation > 0.01 else '✅ 정상'
            }
        
        return violations
    
    def _has_critical_violation(self, violations: Dict) -> bool:
        """심각한 물리 위반 여부 판단"""
        for law_name, info in violations.items():
            if isinstance(info, dict):
                residual = info.get('mean_residual', 0)
                if residual > self.threshold:
                    return True
        return False
    
    def benchmark_speed(self, X_sample: np.ndarray, n_repeats: int = 100) -> Dict:
        """3가지 모드의 추론 속도 벤치마크"""
        results = {}
        modes = ['fast', 'balanced', 'accurate'] if self.pinn else ['fast']
        
        for mode in modes:
            times = []
            for _ in range(n_repeats):
                start = time.time()
                self.predict(X_sample, mode=mode)
                times.append((time.time() - start) * 1000)
            
            results[mode] = {
                'mean_ms': np.mean(times),
                'std_ms': np.std(times),
                'p95_ms': np.percentile(times, 95)
            }
            print(f"Mode {mode:10s}: {results[mode]['mean_ms']:.2f} ± "
                  f"{results[mode]['std_ms']:.2f} ms (p95={results[mode]['p95_ms']:.2f}ms)")
        
        return results

print("✅ HybridSurrogate 클래스 정의 완료")

✅ HybridSurrogate 클래스 정의 완료


# 5. 기존 파이프라인에 PINN 통합 (실행 가이드)

## ⚠️ 전제 조건
- 기존 노트북의 `df` 데이터프레임이 생성되어 있어야 합니다 (Feature Engineering 완료)
- `model_usage`, `model_pf` (XGBoost 모델)가 학습되어 있어야 합니다
- `features` 리스트가 정의되어 있어야 합니다

## 📋 실행 순서
1. 데이터 전처리 (기존 코드 그대로)
2. XGBoost Surrogate 학습 (기존 코드 그대로)
3. 아래 코드 실행: PINN 학습 + Hybrid 통합


In [5]:
# ══════════════════════════════════════════════════════════════
# Step 1: 데이터 준비 (기존 파이프라인 호환)
# ══════════════════════════════════════════════════════════════

def prepare_pinn_data(df, features, target_col='Usage_kWh', test_size=0.2):
    """
    기존 df에서 PINN 학습용 데이터 준비.
    
    기존 파이프라인과의 차이:
    - StandardScaler 사용 (PINN은 정규화에 민감)
    - 시계열 분할 유지 (랜덤 분할 X)
    """
    # 유효한 피처만 선택
    valid_features = [f for f in features if f in df.columns]
    print(f"사용 피처: {len(valid_features)}개")
    
    # 결측치 처리
    target_df = df[valid_features + [target_col]].dropna()
    
    # 시계열 분할 (미래 데이터 누출 방지)
    split_idx = int(len(target_df) * (1 - test_size))
    train_df = target_df.iloc[:split_idx]
    test_df = target_df.iloc[split_idx:]
    
    # 스케일링 (StandardScaler: PINN 안정성 확보)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[valid_features].values.astype(float))
    X_test = scaler.transform(test_df[valid_features].values.astype(float))
    y_train = train_df[target_col].values.astype(float)
    y_test = test_df[target_col].values.astype(float)
    
    # y 스케일러 (별도 - 역변환용)
    y_scaler = StandardScaler()
    y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
    y_test_s = y_scaler.transform(y_test.reshape(-1, 1)).flatten()
    
    print(f"학습 세트: {len(X_train)}개, 검증 세트: {len(X_test)}개")
    
    return {
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train_s, 'y_test': y_test_s,
        'y_train_orig': y_train, 'y_test_orig': y_test,
        'scaler': scaler, 'y_scaler': y_scaler,
        'features': valid_features,
        'train_df': train_df, 'test_df': test_df
    }


# ══════════════════════════════════════════════════════════════
# Step 2: PINN 학습 실행 함수
# ══════════════════════════════════════════════════════════════

def train_pinn_surrogate(
    data_dict,
    physics_registry,
    hidden_dims: List[int] = [128, 64, 32],
    epochs: int = 150,
    lambda_physics: float = 0.1,
    learning_rate: float = 1e-3,
    device: str = None
):
    """
    PINN 서로게이트 학습 메인 함수.
    
    λ_physics 선택 가이드:
    - 데이터 품질 좋음, 물리 법칙 정확 → λ = 0.1~0.3
    - 데이터 희소, 물리 법칙 확실     → λ = 0.3~0.5
    - 물리 법칙 불확실                 → λ = 0.01~0.05
    """
    if device is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    input_dim = data_dict['X_train'].shape[1]
    
    print(f"\n{'='*60}")
    print(f"PINN 초기화 | 입력차원={input_dim} | 구조={hidden_dims}")
    print(f"λ_physics={lambda_physics} | epochs={epochs}")
    print(f"{'='*60}")
    
    # 모델 생성
    pinn_model = PINNSurrogate(
        input_dim=input_dim,
        hidden_dims=hidden_dims,
        physics_registry=physics_registry
    )
    
    # 트레이너 생성
    trainer = PINNTrainer(
        model=pinn_model,
        device=device,
        learning_rate=learning_rate,
        lambda_physics=lambda_physics
    )
    
    # DataLoader 준비
    train_loader, val_loader = trainer.prepare_loaders(
        data_dict['X_train'], data_dict['y_train'],
        data_dict['X_test'], data_dict['y_test']
    )
    
    # 학습
    history = trainer.train(
        train_loader, val_loader,
        epochs=epochs, patience=20
    )
    
    # 학습 곡선 시각화
    trainer.plot_history()
    
    return pinn_model, trainer, history


# ══════════════════════════════════════════════════════════════
# Step 3: 성능 비교 평가
# ══════════════════════════════════════════════════════════════

def evaluate_models(
    data_dict,
    xgb_model,
    pinn_model,
    hybrid_model: Optional[HybridSurrogate] = None
):
    """XGBoost vs PINN vs Hybrid 성능 비교"""
    X_test = data_dict['X_test']
    y_test = data_dict['y_test_orig']
    X_test_raw = data_dict['test_df'][data_dict['features']].values
    
    results = {}
    
    # XGBoost 예측
    xgb_pred = xgb_model.predict(X_test_raw.astype(float))
    results['XGBoost'] = {
        'MAE': mean_absolute_error(y_test, xgb_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, xgb_pred)),
        'R2': r2_score(y_test, xgb_pred),
        'pred': xgb_pred
    }
    
    # PINN 예측 (스케일 역변환)
    if pinn_model is not None:
        pinn_pred_s = pinn_model.predict_numpy(X_test)
        pinn_pred = data_dict['y_scaler'].inverse_transform(
            pinn_pred_s.reshape(-1, 1)
        ).flatten()
        results['PINN'] = {
            'MAE': mean_absolute_error(y_test, pinn_pred),
            'RMSE': np.sqrt(mean_squared_error(y_test, pinn_pred)),
            'R2': r2_score(y_test, pinn_pred),
            'pred': pinn_pred
        }
    
    # 결과 출력
    print("\n" + "="*55)
    print(f"{'Model':<15} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
    print("-"*55)
    for name, m in results.items():
        print(f"{name:<15} {m['MAE']:>8.3f} {m['RMSE']:>8.3f} {m['R2']:>8.4f}")
    print("="*55)
    
    # 시각화: 예측 vs 실측
    fig, axes = plt.subplots(1, len(results), figsize=(7*len(results), 5))
    if len(results) == 1:
        axes = [axes]
    
    sample_n = min(500, len(y_test))
    idx = np.random.choice(len(y_test), sample_n, replace=False)
    
    for ax, (name, m) in zip(axes, results.items()):
        ax.scatter(y_test[idx], m['pred'][idx], alpha=0.4, s=15, color='dodgerblue')
        lim = [min(y_test.min(), m['pred'].min()), max(y_test.max(), m['pred'].max())]
        ax.plot(lim, lim, 'r--', lw=2, label='Perfect')
        ax.set_title(f"{name}\nMAE={m['MAE']:.3f}, R²={m['R2']:.4f}")
        ax.set_xlabel('Actual (kWh)')
        ax.set_ylabel('Predicted (kWh)')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('Model Performance: Actual vs Predicted', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    return results

print("✅ 통합 실행 함수 정의 완료")

✅ 통합 실행 함수 정의 완료


# 6. λ_physics 튜닝 실험 — 속도 vs 정확도 트레이드오프 분석

In [6]:
def run_lambda_experiment(
    data_dict,
    physics_registry,
    lambdas: List[float] = [0.0, 0.01, 0.05, 0.1, 0.3, 0.5],
    epochs: int = 80
) -> pd.DataFrame:
    """
    다양한 λ_physics 값으로 실험하여 최적값 탐색.
    
    실험 결과 해석:
    - λ=0.0:  순수 데이터 학습 (PINN 아님, 기준선)
    - 최적 λ: Val MAE 최소 & Physics Loss 안정적인 값
    - 큰 λ:   물리 제약 과도 → 데이터 피팅 포기
    """
    results = []
    
    for lam in lambdas:
        print(f"\n{'─'*40}")
        print(f"실험: λ_physics = {lam}")
        
        model = PINNSurrogate(
            input_dim=data_dict['X_train'].shape[1],
            hidden_dims=[64, 32],
            physics_registry=physics_registry
        )
        trainer = PINNTrainer(model, lambda_physics=lam, learning_rate=1e-3)
        train_l, val_l = trainer.prepare_loaders(
            data_dict['X_train'], data_dict['y_train'],
            data_dict['X_test'], data_dict['y_test'],
            batch_size=512
        )
        trainer.train(train_l, val_l, epochs=epochs, patience=15, verbose=False)
        
        # 평가
        pred_s = model.predict_numpy(data_dict['X_test'])
        pred = data_dict['y_scaler'].inverse_transform(pred_s.reshape(-1,1)).flatten()
        
        results.append({
            'lambda': lam,
            'val_mae': mean_absolute_error(data_dict['y_test_orig'], pred),
            'val_r2': r2_score(data_dict['y_test_orig'], pred),
            'final_physics_loss': trainer.history['physics'][-1],
            'final_data_loss': trainer.history['data'][-1]
        })
        print(f"  → Val MAE={results[-1]['val_mae']:.3f}, R²={results[-1]['val_r2']:.4f}, "
              f"Physics Loss={results[-1]['final_physics_loss']:.4f}")
    
    df_res = pd.DataFrame(results)
    
    # 시각화
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    axes[0].plot(df_res['lambda'], df_res['val_mae'], marker='o', color='dodgerblue', lw=2)
    axes[0].set_title('λ vs Validation MAE\n(낮을수록 좋음)')
    axes[0].set_xlabel('λ_physics')
    axes[0].set_ylabel('MAE (kWh)')
    axes[0].axvline(df_res.loc[df_res['val_mae'].idxmin(), 'lambda'], 
                    color='red', ls='--', alpha=0.7, label='Best λ')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(df_res['lambda'], df_res['final_physics_loss'], marker='s', color='tomato', lw=2)
    axes[1].set_title('λ vs Physics Loss\n(낮을수록 물리 정합성 높음)')
    axes[1].set_xlabel('λ_physics')
    axes[1].set_ylabel('Physics Loss')
    axes[1].grid(True, alpha=0.3)
    
    axes[2].scatter(df_res['val_mae'], df_res['final_physics_loss'], 
                    c=df_res['lambda'], cmap='viridis', s=100)
    for _, row in df_res.iterrows():
        axes[2].annotate(f"λ={row['lambda']}", (row['val_mae'], row['final_physics_loss']),
                        textcoords='offset points', xytext=(5,5), fontsize=9)
    axes[2].set_title('트레이드오프: MAE vs Physics Loss\n(좌하단이 이상적)')
    axes[2].set_xlabel('Val MAE (kWh)')
    axes[2].set_ylabel('Physics Loss')
    axes[2].grid(True, alpha=0.3)
    
    best_lambda = df_res.loc[df_res['val_mae'].idxmin(), 'lambda']
    plt.suptitle(f'λ_physics Sensitivity Analysis | 최적 λ = {best_lambda}', 
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    return df_res, best_lambda

print("✅ λ 튜닝 실험 함수 정의 완료")

# ──── 실제 실행 예시 ─────────────────────────────────────────
# 주의: 기존 파이프라인에서 df, features, model_usage가 정의된 후 실행
# 
# if TORCH_AVAILABLE and 'df' in dir():
#     features_pinn = [
#         'Hour', 'Is_Weekend', 'Month_sin', 'Month_cos',
#         'DayOfWeek_sin', 'DayOfWeek_cos',
#         'Motor_Operating_Rate', 'Capacitor_Operating_Rate',
#         'Is_Operating_Flag', 'Motor_Moving_Avg_1h_std'
#     ]
#     
#     data = prepare_pinn_data(df, features_pinn)
#     
#     # λ 탐색
#     lambda_results, best_lam = run_lambda_experiment(data, physics)
#     
#     # 최적 λ로 PINN 학습
#     pinn_model, trainer, history = train_pinn_surrogate(
#         data, physics, lambda_physics=best_lam, epochs=150
#     )
#     
#     # 성능 비교
#     eval_results = evaluate_models(data, model_usage, pinn_model)

✅ λ 튜닝 실험 함수 정의 완료


# 7. 디지털 트윈 물리 검증 레이어 (실시간 모니터링)

In [7]:
class PhysicsMonitor:
    """
    실시간 물리 법칙 위반 모니터링.
    
    디지털 트윈 운영 중 센서 데이터를 받아:
    1. 물리 법칙 위반 여부 실시간 점검
    2. 이상 감지 시 알람 생성
    3. 이력 관리 및 드리프트 감지
    
    현장 활용:
    - OPC-UA / MQTT 스트림에 연결하여 실시간 감시
    - 이상 시 DCS/SCADA 알람 연동 가능
    """
    def __init__(self, physics_registry: PhysicsRegistry, window_size: int = 96):
        self.physics = physics_registry
        self.window_size = window_size  # 모니터링 윈도우 (96 = 15분 * 96 = 1일)
        self.violation_history = []
        self.alert_log = []
    
    def check_row(self, row: pd.Series) -> Dict:
        """단일 데이터 포인트 물리 검증"""
        report = {'timestamp': row.get('date', 'N/A'), 'violations': {}, 'alert': False}
        
        # 전력 삼각형 검증 (직접 계산)
        if all(c in row.index for c in ['Usage_kWh', 'Lagging_Current_Reactive_Power_kVarh',
                                          'Leading_Current_Reactive_Power_kVarh', 'Apparent_Power']):
            P = row['Usage_kWh']
            Q_net = (row['Lagging_Current_Reactive_Power_kVarh'] - 
                     row['Leading_Current_Reactive_Power_kVarh'])
            S_calc = np.sqrt(P**2 + Q_net**2 + 1e-8)
            S_meas = row['Apparent_Power']
            residual = abs(S_calc - S_meas) / (S_meas + 1e-8)
            
            report['violations']['PowerTriangle'] = {
                'residual': residual,
                'status': '⚠️' if residual > 0.05 else '✅'
            }
            if residual > 0.1:
                report['alert'] = True
                report['alert_msg'] = f"전력 삼각형 위반 (잔차={residual:.3f}) - 센서 오류 의심"
        
        # 역률 범위 검증
        if 'PF_Physical' in row.index:
            pf = row['PF_Physical']
            if pf < 0 or pf > 1:
                report['violations']['PowerFactor'] = {
                    'value': pf, 'status': '⚠️'
                }
                report['alert'] = True
                report['alert_msg'] = f"역률 범위 이탈: PF={pf:.3f}"
        
        self.violation_history.append(report)
        if report['alert']:
            self.alert_log.append(report)
        
        return report
    
    def batch_check(self, df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
        """데이터프레임 전체 물리 검증"""
        reports = []
        for _, row in df.iterrows():
            r = self.check_row(row)
            reports.append({
                'timestamp': r['timestamp'],
                'alert': r['alert'],
                'power_triangle_residual': r['violations'].get(
                    'PowerTriangle', {}).get('residual', np.nan),
                'pf_status': r['violations'].get('PowerFactor', {}).get('status', '✅')
            })
        
        result_df = pd.DataFrame(reports)
        
        if verbose:
            n_alerts = result_df['alert'].sum()
            alert_rate = n_alerts / len(result_df) * 100
            print(f"\n{'='*50}")
            print(f"물리 검증 결과 | 총 {len(result_df):,}개 포인트")
            print(f"  알람 발생: {n_alerts:,}개 ({alert_rate:.1f}%)")
            print(f"  전력삼각형 평균 잔차: {result_df['power_triangle_residual'].mean():.4f}")
            print(f"{'='*50}")
            
            # 시각화
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            
            axes[0].plot(result_df['power_triangle_residual'].values, 
                        color='dodgerblue', alpha=0.7, lw=0.8)
            axes[0].axhline(0.05, color='orange', ls='--', label='Warning (0.05)')
            axes[0].axhline(0.10, color='red', ls='--', label='Alert (0.10)')
            axes[0].fill_between(range(len(result_df)), 
                                result_df['power_triangle_residual'].values,
                                where=result_df['alert'], color='red', alpha=0.3, label='Alert Zone')
            axes[0].set_title('실시간 전력 삼각형 잔차 모니터링')
            axes[0].set_xlabel('Sample Index')
            axes[0].set_ylabel('Normalized Residual')
            axes[0].legend()
            axes[0].grid(True, alpha=0.3)
            
            # 알람 빈도 시각화 (시간대별)
            if 'Hour' in df.columns:
                df_check = df.copy()
                df_check['alert'] = result_df['alert'].values
                hourly_alert = df_check.groupby('Hour')['alert'].mean() * 100
                axes[1].bar(hourly_alert.index, hourly_alert.values, color='tomato', alpha=0.8)
                axes[1].set_title('시간대별 물리 위반 빈도 (%)')
                axes[1].set_xlabel('Hour')
                axes[1].set_ylabel('Alert Rate (%)')
                axes[1].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
        
        return result_df


# ──── 실행 예시 ──────────────────────────────────────────────
# if 'df' in dir() and 'Apparent_Power' in df.columns:
#     monitor = PhysicsMonitor(physics_registry=physics)
#     check_results = monitor.batch_check(df.head(1000))
    
print("✅ PhysicsMonitor 클래스 정의 완료")

✅ PhysicsMonitor 클래스 정의 완료


# 8. 새로운 공정 적용 가이드 (범용화 인터페이스)

## 🏭 공정 어댑터 템플릿
"""
새로운 현장에 적용 시, 아래 2가지만 구현하면 됩니다:
1. 해당 공정의 PhysicsLaw 구현체
2. ProcessAdapter.get_physics_registry() 메서드

나머지 파이프라인 (PINN 학습, Hybrid 통합, 최적화)은 그대로 재사용
"""


In [8]:
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class ProcessConfig:
    """
    공정 설정 데이터 클래스.
    
    새 공정 적용 시 이 파일만 수정하면 됩니다.
    """
    name: str                       # 공정명
    target_variables: List[str]     # 예측 목표 변수
    control_variables: List[str]    # 최적화 제어 변수
    state_variables: List[str]      # 상태 관측 변수
    constraints: Dict               # 물리/운영 제약
    sampling_interval_min: int = 15 # 샘플링 주기 (분)


class ProcessAdapter(ABC):
    """
    공정 어댑터 추상 기반 클래스.
    
    구현 필수 메서드:
    - get_physics_registry(): 해당 공정의 물리 법칙 레지스트리 반환
    - get_feature_template(): 피처 엔지니어링 파이프라인 반환
    - get_process_config(): 공정 설정 반환
    """
    
    @abstractmethod
    def get_physics_registry(self) -> PhysicsRegistry:
        pass
    
    @abstractmethod
    def get_process_config(self) -> ProcessConfig:
        pass
    
    def build_pipeline(self) -> Dict:
        """전체 PINN 파이프라인 빌더"""
        config = self.get_process_config()
        registry = self.get_physics_registry()
        
        return {
            'config': config,
            'physics': registry,
            'pinn_model': None,  # train_pinn_surrogate()로 채워짐
            'hybrid': None,      # HybridSurrogate()로 채워짐
            'monitor': PhysicsMonitor(registry)
        }


# ══════════════════════════════════════════════════════════════
# [현재 프로젝트] 전력 계통 어댑터
# ══════════════════════════════════════════════════════════════

class SteelPowerAdapter(ProcessAdapter):
    """
    제철소 전력 계통 어댑터 (현재 프로젝트).
    
    이 클래스가 현재 프로젝트의 구체화 구현체입니다.
    새 공정 적용 시 이 클래스를 참고하여 새로 작성하세요.
    """
    def get_physics_registry(self) -> PhysicsRegistry:
        return PhysicsRegistry.power_system()
    
    def get_process_config(self) -> ProcessConfig:
        return ProcessConfig(
            name="Steel_Power_System",
            target_variables=['Usage_kWh', 'PF_Physical'],
            control_variables=['Motor_Operating_Rate', 'Capacitor_Operating_Rate'],
            state_variables=['Lagging_Current_Reactive_Power_kVarh',
                            'Leading_Current_Reactive_Power_kVarh',
                            'CO2_ppm', 'PSI'],
            constraints={
                'PF_Physical': {'min': 0.9, 'max': 1.0},   # 한전 역률 기준
                'PSI': {'max': 85},                          # 공정 스트레스 한계
                'Motor_Operating_Rate': {'min': 0, 'max': 100},
                'Capacitor_Operating_Rate': {'min': 0, 'max': 100}
            },
            sampling_interval_min=15
        )


# ══════════════════════════════════════════════════════════════
# [확장 예시] 화학 공정 어댑터 (미구현 - 템플릿)
# ══════════════════════════════════════════════════════════════

class ChemicalProcessAdapter(ProcessAdapter):
    """
    화학 공정 어댑터 템플릿.
    
    구현 시 아래 PhysicsLaw를 작성하고 registry에 등록하면 됩니다:
    
    class MassBalanceLaw(PhysicsLaw):
        def residual(self, inputs, outputs):
            F_in = inputs['flow_in']
            F_out = inputs['flow_out']
            dM_dt = inputs.get('mass_accumulation', 0)
            return F_in - F_out - dM_dt  # 질량 보존 위반량
    
    class ArrheniusLaw(PhysicsLaw):
        def __init__(self, Ea: float, A: float, R: float = 8.314):
            self.Ea, self.A, self.R = Ea, A, R
        def residual(self, inputs, outputs):
            T = inputs['temperature']
            k_measured = outputs['reaction_rate']
            k_calc = self.A * torch.exp(-self.Ea / (self.R * T))
            return (k_measured - k_calc) / (k_calc + 1e-8)
    """
    def get_physics_registry(self) -> PhysicsRegistry:
        # TODO: 구현 필요
        return PhysicsRegistry.chemical_process()
    
    def get_process_config(self) -> ProcessConfig:
        return ProcessConfig(
            name="Chemical_Process",
            target_variables=['yield', 'temperature_out'],
            control_variables=['flow_rate', 'temperature_set', 'pressure'],
            state_variables=['concentration', 'pH', 'viscosity'],
            constraints={
                'temperature_out': {'min': 50, 'max': 200},
                'pressure': {'min': 1, 'max': 10}
            }
        )


# ── 현재 프로젝트 어댑터 초기화 ─────────────────────────────
current_adapter = SteelPowerAdapter()
pipeline = current_adapter.build_pipeline()

print("\n✅ 공정 어댑터 초기화 완료")
print(f"공정명: {pipeline['config'].name}")
print(f"제어 변수: {pipeline['config'].control_variables}")
print(f"물리 제약: {list(pipeline['config'].constraints.keys())}")


🔌 전력 계통 PhysicsRegistry 초기화 중...
  ✅ 등록: PowerTriangle (weight=1.0)
  ✅ 등록: PowerFactor (weight=0.5)
  ✅ 등록: EnergySmoothing (weight=0.3)
  ✅ 등록: KEPCOCapacitorStrategy (weight=0.4)

✅ 공정 어댑터 초기화 완료
공정명: Steel_Power_System
제어 변수: ['Motor_Operating_Rate', 'Capacitor_Operating_Rate']
물리 제약: ['PF_Physical', 'PSI', 'Motor_Operating_Rate', 'Capacitor_Operating_Rate']


# 9. Quick Start — 전체 파이프라인 원클릭 실행

In [9]:
"""
이 셀 하나로 기존 파이프라인 + PINN 전체를 실행합니다.

사전 조건:
- df (Feature Engineering 완료 데이터프레임)
- model_usage (기존 XGBoost 사용량 예측 모델)
- features (피처 리스트)
- PyTorch 설치
"""

def run_full_pinn_pipeline(df, xgb_model, features, lambda_physics=0.1, epochs=100):
    """
    PINN 통합 파이프라인 원클릭 실행.
    
    Returns:
        hybrid_model: HybridSurrogate (실제 현장 배포용)
        eval_results: 모델 성능 비교
        monitor: PhysicsMonitor (실시간 감시용)
    """
    print("\n" + "🚀"*30)
    print("PINN 통합 디지털 트윈 파이프라인 시작")
    print("🚀"*30)
    
    if not TORCH_AVAILABLE:
        print("❌ PyTorch가 없어 PINN을 실행할 수 없습니다.")
        print("   pip install torch 후 재실행하세요.")
        return None, None, None
    
    # Step 1: 데이터 준비
    print("\n[Step 1/5] 데이터 준비...")
    data = prepare_pinn_data(df, features)
    
    # Step 2: 물리 레지스트리
    print("\n[Step 2/5] 물리 법칙 레지스트리...")
    physics_reg = PhysicsRegistry.power_system()
    
    # Step 3: PINN 학습
    print(f"\n[Step 3/5] PINN 학습 (λ={lambda_physics})...")
    pinn_model, trainer, _ = train_pinn_surrogate(
        data, physics_reg,
        lambda_physics=lambda_physics,
        epochs=epochs
    )
    
    # Step 4: Hybrid 통합
    print("\n[Step 4/5] Hybrid Surrogate 통합...")
    hybrid = HybridSurrogate(
        xgb_model=xgb_model,
        pinn_model=pinn_model,
        scaler=data['scaler'],
        physics_registry=physics_reg,
        correction_threshold=0.1
    )
    
    # Step 5: 평가
    print("\n[Step 5/5] 성능 비교 평가...")
    eval_results = evaluate_models(data, xgb_model, pinn_model, hybrid)
    
    # 속도 벤치마크
    print("\n[Bonus] 추론 속도 벤치마크...")
    sample_X = data['test_df'][data['features']].values[:10]
    hybrid.benchmark_speed(sample_X, n_repeats=50)
    
    # 물리 모니터
    monitor = PhysicsMonitor(physics_reg)
    
    print("\n" + "✅"*30)
    print("파이프라인 완료! HybridSurrogate를 현장 배포에 사용하세요.")
    print("✅"*30)
    
    return hybrid, eval_results, monitor


# ──── 실행 (기존 파이프라인 완료 후 아래 주석 해제) ────────────
# hybrid_model, results, monitor = run_full_pinn_pipeline(
#     df=df,
#     xgb_model=model_usage,
#     features=features,
#     lambda_physics=0.1,
#     epochs=100
# )

print("\n📋 사용법: 기존 파이프라인 실행 후 위 run_full_pinn_pipeline() 호출")
print("   현장 배포 시 hybrid_model.predict(X, mode='balanced') 사용")


📋 사용법: 기존 파이프라인 실행 후 위 run_full_pinn_pipeline() 호출
   현장 배포 시 hybrid_model.predict(X, mode='balanced') 사용


# 10. 생산량 기반 보상항 설계 (CO2 파생 지표)
## 문제: 기존 목적함수는 패널티만 존재
```
기존: J = cost + λ_psi × PSI + m_rate × 0.001 앵커
```
퇴화 해 위험: 보상 없이 비용만 최소화하면
"공정을 완전히 멈추는 것"이 수학적으로 최적해가 될 수 있음.

## 해결: CO2 파생 생산 지표 → 보상항 추가
```
수정: J = cost + λ_psi × PSI
         − λ_epe  × EPE    ← 단위 전력당 생산성 (에너지 효율 극대화)
         − λ_cont × OCS    ← 가동 연속성 (생산 중단 패널티 방지)
```

## CO2를 생산 프록시로 쓰는 근거와 한계
| 항목 | 내용 |
|------|------|
| 근거 | 철강 공정에서 CO2↑ = 연료 연소 = 생산 활동 중 |
| 데이터 근거 | Is_Operating_Flag=2 구간에서 CO2_ppm 유의미하게 상승 확인 |
| 한계 | 연료 CO2 + 공정 CO2 혼재 → Is_Operating_Flag로 필터링 필수 |
| 보정 방법 | 비가동(Flag=0) 구간 CO2를 배경값으로 차감 |


## 10.1 CO2 기반 생산 파생 지표 계산

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

class ProductionProxyBuilder:
    """
    CO2 센서 데이터로부터 생산 파생 지표 3종 생성.

    ──────────────────────────────────────────────────────────
    [센서 특성 기반 설계 단순화]

    이 데이터셋의 CO2 센서는 민감도가 낮아 공장 비가동 시
    실제로 0 또는 0에 가까운 값을 출력한다.

    따라서:
        CO2_observed ≈ 0              (비가동 시)
        CO2_observed ≈ CO2_process    (가동 시: 연료 연소 + 반응 부산물)

    → 배경값(CO2_background) 추정·차감 단계가 불필요하다.
      오히려 배경값을 빼면 이미 0인 구간에서 음수 발생 → clip으로
      정보 손실이 생기는 역효과가 있다.

    → CO2_observed 자체를 공정 기여 신호로 직접 사용한다.

    ──────────────────────────────────────────────────────────
    [CO2 신호 물리적 의미]

        CO2 (가동 중) = CO2_fuel        (연료 연소 → 생산량에 비례)
                      + CO2_reaction    (제철 반응 부산물 → 생산량에 비례)

    두 성분 모두 생산 활동에 비례 → CO2 전체가 유효한 생산 프록시

    ──────────────────────────────────────────────────────────
    [생성 지표 3종]

    ① PPI (Production Proxy Index)
       = CO2_observed / CO2_std_operating
       해석: 가동 중 CO2 표준편차 기준 정규화된 생산 강도
       범위: 0 이상 (비가동 시 자동으로 0 또는 0에 근접)

    ② EPE (Energy Production Efficiency)
       = PPI / Usage_kWh
       해석: 단위 전력 투입 대비 생산 기여량
             높을수록 에너지를 생산에 효율적으로 사용
       최적화 목표: 최대화 (보상항)

    ③ OCS (Operation Continuity Score)
       = rolling_mean(Is_Operating_Flag > 0, window=4)
       해석: 최근 1시간(15분×4) 가동 비율
       최적화 목표: 최대화 (보상항)

    [통합 보상 지수]
    PRI = w_epe × EPE_norm + w_ocs × OCS
          (가중치는 CO2-Usage 단조 관계 강도에 따라 자동 조정)
    ──────────────────────────────────────────────────────────
    """

    def __init__(self, co2_col="CO2_ppm", op_flag_col="Is_Operating_Flag",
                 usage_col="Usage_kWh", window=4):
        self.co2_col   = co2_col
        self.op_col    = op_flag_col
        self.usage_col = usage_col
        self.window    = window
        # fit() 결과 파라미터
        self.co2_std   = None
        self.epe_cap   = None
        self.w_epe     = 0.6    # PRI 내 EPE 가중치 (단조 검증 후 자동 조정)
        self.w_ocs     = 0.4    # PRI 내 OCS 가중치
        self.spearman_r = None  # 단조 관계 강도 기록

    # ── Step 1: 스케일 파라미터 추정 ──────────────────────────
    def fit(self, df: pd.DataFrame):
        """
        가동 구간 CO2 표준편차 추정 + CO2-Usage 단조 관계 검증.

        배경값 추정 없음 (센서가 비가동 시 0 출력 → 불필요).
        반드시 학습 데이터(train_df)로만 fit → 미래 누출 방지.
        """
        op_mask = (df[self.op_col] == 2)

        # ── CO2 정규화 기준: 가동 구간 표준편차 ───────────────
        co2_operating = df.loc[op_mask, self.co2_col]
        if op_mask.sum() > 10:
            self.co2_std = co2_operating.std()
        else:
            self.co2_std = df[self.co2_col].std()
            print("  ⚠️  가동 구간 샘플 부족 → 전체 표준편차 사용")

        if self.co2_std < 1e-3:
            self.co2_std = 1.0  # 분산 0 방지

        # ── EPE 이상값 상한 ─────────────────────────────────
        usage_pos = df[self.usage_col].clip(lower=1.0)
        epe_raw   = df[self.co2_col] / usage_pos
        self.epe_cap = epe_raw[op_mask].quantile(0.99) if op_mask.sum() > 10                        else epe_raw.quantile(0.99)
        if self.epe_cap < 1e-6:
            self.epe_cap = 1.0

        # ── CO2-Usage 단조 관계 검증 (PRI 가중치 자동 결정) ──
        if op_mask.sum() > 30:
            r_s, p_s = stats.spearmanr(
                df.loc[op_mask, self.co2_col],
                df.loc[op_mask, self.usage_col]
            )
            self.spearman_r = r_s

            if r_s > 0.4 and p_s < 0.05:
                self.w_epe, self.w_ocs = 0.6, 0.4
                flag = "✅  강함"
            elif r_s > 0.2 and p_s < 0.05:
                self.w_epe, self.w_ocs = 0.5, 0.5
                flag = "⚡  중간"
            else:
                self.w_epe, self.w_ocs = 0.3, 0.7
                flag = "⚠️   약함"

            print(f"  {flag}  CO2-Usage 단조 관계 (Spearman r={r_s:.3f}, p={p_s:.4f})")
            print(f"         → PRI 가중치 자동 설정: EPE {self.w_epe:.0%} / OCS {self.w_ocs:.0%}")
        else:
            print("  ⚠️  가동 구간 샘플 부족 → 단조 검증 생략, 기본 가중치 사용")

        print(f"\n✅ fit 완료")
        print(f"   CO2 σ (가동 중): {self.co2_std:.2f} ppm")
        print(f"   EPE 상한 (99%):  {self.epe_cap:.4f}")
        return self

    # ── Step 2: 지표 생성 ────────────────────────────────────
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        CO2_observed 를 직접 사용해 3종 지표 생성.
        배경값 차감 없음 (센서 특성상 불필요).
        """
        assert self.co2_std is not None, "fit()을 먼저 실행하세요."
        out = df.copy()

        # ── ① PPI (Production Proxy Index) ───────────────────
        # CO2_observed 직접 정규화 (배경값 차감 없음)
        out["PPI"] = out[self.co2_col] / self.co2_std

        # CO2-Usage 단조 관계 약할 때: rolling smoothing으로 노이즈 완화
        if self.spearman_r is not None and self.spearman_r < 0.2:
            out["PPI"] = out["PPI"].rolling(window=4, min_periods=1).mean()
            print("  ℹ️  단조 관계 약함 → PPI에 4-step rolling mean 적용")

        # ── ② EPE (Energy Production Efficiency) ─────────────
        usage_pos  = out[self.usage_col].clip(lower=1.0)
        out["EPE"] = (out["PPI"] / usage_pos).clip(upper=self.epe_cap)

        # 비가동 구간 EPE = 0 (CO2≈0 이므로 PPI≈0 → 자연스럽게 0에 가깝지만 명시)
        out.loc[out[self.op_col] == 0, "EPE"] = 0.0

        # ── ③ OCS (Operation Continuity Score) ───────────────
        is_on      = (out[self.op_col] > 0).astype(float)
        out["OCS"] = is_on.rolling(window=self.window, min_periods=1).mean()

        # ── PRI (통합 보상 지수) ──────────────────────────────
        # 가중치는 fit() 에서 단조 관계 강도에 따라 자동 결정됨
        epe_norm   = out["EPE"] / (out["EPE"].max() + 1e-8)
        out["PRI"] = self.w_epe * epe_norm + self.w_ocs * out["OCS"]

        op_mask = out[self.op_col] == 2
        print(f"✅ 파생 지표 생성 완료")
        print(f"   PPI 평균 (가동 중): {out.loc[op_mask, 'PPI'].mean():.3f}")
        print(f"   EPE 평균 (가동 중): {out.loc[op_mask, 'EPE'].mean():.4f}")
        print(f"   OCS 전체 평균:      {out['OCS'].mean():.3f}")
        print(f"   PRI 가중치:         EPE {self.w_epe:.0%} / OCS {self.w_ocs:.0%}")
        return out

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.fit(df).transform(df)

    # ── Step 3: 유효성 시각화 ────────────────────────────────
    def validate(self, df: pd.DataFrame, figsize=(16, 12)):
        """
        4종 유효성 검증 차트.

        [확인 포인트]
        ① PPI 분포: 비가동(0)과 가동(양수)이 명확히 분리되는가
        ② CO2 vs Usage (가동 중): 단조 상관 시각적 확인
        ③ OCS 시계열: 가동 상태와 동기화 확인
        ④ PRI 분포: 보상 지수의 형태 (0 근처 집중 여부 확인)
        """
        assert "PPI" in df.columns, "transform()을 먼저 실행하세요."
        op_mask  = df[self.op_col] == 2
        off_mask = df[self.op_col] == 0

        fig, axes = plt.subplots(2, 2, figsize=figsize)
        palette = {0: "silver", 1: "orange", 2: "dodgerblue"}

        # ① PPI 분포 (상태별)
        ax = axes[0, 0]
        for flag, grp in df.groupby(self.op_col):
            if len(grp) > 5:
                sns.kdeplot(grp["PPI"], ax=ax, fill=True, alpha=0.4,
                            label=f"Flag={flag}", color=palette.get(flag, "gray"))
        zero_pct = (df["PPI"] < 0.01).mean() * 100
        ax.set_title(f"① PPI 분포 (상태별 분리 확인)\n"
                     f"0에 가까운 비율: {zero_pct:.1f}% (비가동 구간)",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("PPI")
        ax.legend()
        ax.grid(True, alpha=0.3)

        # ② CO2 vs Usage (단조 관계 핵심 검증)
        ax = axes[0, 1]
        ax.scatter(df.loc[op_mask,  self.usage_col],
                   df.loc[op_mask,  self.co2_col],
                   alpha=0.3, s=8, color="dodgerblue", label="가동 중 (Flag=2)")
        ax.scatter(df.loc[off_mask, self.usage_col],
                   df.loc[off_mask, self.co2_col],
                   alpha=0.2, s=8, color="silver", label="비가동 (Flag=0)")
        r_label = f"r={self.spearman_r:.3f}" if self.spearman_r else "미계산"
        status  = "✅ 유효" if (self.spearman_r or 0) > 0.2 else "⚠️ 약함"
        ax.set_title(f"② CO2 vs 전력 사용량 (단조 관계 검증)\n"
                     f"Spearman {r_label} {status}",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("Usage_kWh")
        ax.set_ylabel("CO2_ppm")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

        # ③ OCS 시계열 (1주일 샘플)
        ax    = axes[1, 0]
        sample = df.iloc[:min(192*7, len(df))]
        ax2   = ax.twinx()
        ax.plot(range(len(sample)), sample["OCS"],
                color="green", lw=1.5, label="OCS")
        ax2.step(range(len(sample)), sample[self.op_col],
                 color="red", alpha=0.3, where="post", label="Op Flag")
        ax.set_title("③ OCS 시계열 (가동 상태와 동기화 확인)", fontsize=12, fontweight="bold")
        ax.set_ylabel("OCS", color="green")
        ax2.set_ylabel("Is_Operating_Flag", color="red")
        ax.legend(loc="upper left", fontsize=9)
        ax.grid(True, alpha=0.3)

        # ④ PRI 분포
        ax = axes[1, 1]
        ax.hist(df["PRI"], bins=50, color="dodgerblue", alpha=0.7, edgecolor="white")
        ax.axvline(df["PRI"].median(), color="red", ls="--",
                   label=f"중앙값: {df['PRI'].median():.3f}")
        ax.set_title(f"④ PRI 분포 (보상 지수)\n"
                     f"EPE {self.w_epe:.0%} + OCS {self.w_ocs:.0%}",
                     fontsize=12, fontweight="bold")
        ax.set_xlabel("PRI")
        ax.legend()
        ax.grid(True, alpha=0.3)

        plt.suptitle("CO2 기반 생산 파생 지표 유효성 검증", fontsize=15, y=1.01)
        plt.tight_layout()
        plt.show()

        # 수치 요약
        print("\n=== 지표 요약 (Is_Operating_Flag별 평균) ===")
        print(df.groupby(self.op_col)[["PPI", "EPE", "OCS", "PRI"]].mean().round(4))

        # λ 조정 권고
        print("\n=== λ 가중치 권고 ===")
        r = self.spearman_r or 0
        if r > 0.4:
            print(f"  CO2-Usage 상관 강함 (r={r:.3f})")
            print(f"  → EPE 보상 신뢰 가능. 현재 λ_epe 비중 유지.")
        elif r > 0.2:
            print(f"  CO2-Usage 상관 중간 (r={r:.3f})")
            print(f"  → PRI = 0.5×EPE + 0.5×OCS 균형 유지 권장.")
        else:
            print(f"  CO2-Usage 상관 약함 (r={r:.3f})")
            print(f"  → OCS 중심으로 운영: λ_cont 높이고 λ_epe 낮추기.")
            print(f"  → run_reward_lambda_sensitivity()로 재탐색 권장.")


# ──── 실행 ──────────────────────────────────────────────────
# if "df" in dir():
#     split_idx = int(len(df) * 0.8)
#     prod_builder = ProductionProxyBuilder()
#     prod_builder.fit(df.iloc[:split_idx])   # train_df 로만 fit
#     df = prod_builder.transform(df)         # 전체 적용
#     prod_builder.validate(df)

print("✅ ProductionProxyBuilder (최종판) 정의 완료")
print()
print("변경 사항:")
print("  - 배경값(CO2_background) 추정·차감 제거")
print("    (센서가 비가동 시 0 출력 → 배경값 = 0으로 간주)")
print("  - CO2_observed 직접 PPI 계산")
print("  - 단조 관계 강도에 따라 PRI 가중치 자동 결정")
print("    r > 0.4 → EPE 60% / OCS 40%")
print("    r > 0.2 → EPE 50% / OCS 50%")
print("    r ≤ 0.2 → EPE 30% / OCS 70%")


✅ ProductionProxyBuilder (최종판) 정의 완료

변경 사항:
  - 배경값(CO2_background) 추정·차감 제거
    (센서가 비가동 시 0 출력 → 배경값 = 0으로 간주)
  - CO2_observed 직접 PPI 계산
  - 단조 관계 강도에 따라 PRI 가중치 자동 결정
    r > 0.4 → EPE 60% / OCS 40%
    r > 0.2 → EPE 50% / OCS 50%
    r ≤ 0.2 → EPE 30% / OCS 70%


## 10.2 목적함수 수정 — 보상항 통합

In [11]:
"""
[기존 목적함수 구조 분석]

기존 objective(trial):
    cost  = usage × current_price          ← 에너지 비용
    cost += m_rate × 0.001                 ← 모터 과가동 앵커 (퇴화 해 방지용)
    cost += PF_penalty × 1,000,000         ← 역률 위반 패널티
    cost -= PF_reward × ...                ← 역률 개선 보상 (일부 있음)

문제점:
    1. m_rate × 0.001 앵커: 모터를 낮추면 항상 유리 → 퇴화 해 경향
    2. 생산량 개념 없음: '공정 멈춤 = 최적'이 수학적으로 성립
    3. 가동 중단 시 비용이 0원 → 실제 기회비용 미반영

[수정 목적함수 구조]

J = cost                                  ← 에너지 비용 (기존 유지)
  + λ_psi  × PSI_penalty                 ← 공정 스트레스 (기존 유지)
  + λ_pf   × PF_penalty                  ← 역률 페널티 (기존 유지)
  − λ_epe  × EPE_reward                  ← 에너지 생산 효율 보상 [추가]
  − λ_cont × OCS_reward                  ← 가동 연속성 보상 [추가]

λ 가중치 설정 기준 (규모 통일 원칙):
    패널티와 보상의 기여 규모를 맞춰야 최적화가 균형잡힘
    - λ_epe:  EPE 보상이 에너지 비용의 최대 20~30% 수준이 되도록 설정
    - λ_cont: OCS 보상이 m_rate 앵커를 대체할 수준 (0.001 × 100 ≈ 0.1)
"""

import numpy as np
import optuna

# ── 보상 스케일 맞춤 헬퍼 ─────────────────────────────────────
class RewardScaler:
    """
    학습 데이터 기준으로 보상항의 스케일을 에너지 비용과 통일.

    사용법:
        scaler = RewardScaler()
        scaler.fit(df, current_price_avg=50.0)
        lambda_epe, lambda_cont = scaler.suggest_lambdas()
    """
    def __init__(self):
        self.cost_mean  = None
        self.epe_mean   = None
        self.ocs_mean   = None

    def fit(self, df, current_price_avg: float = 50.0):
        """학습 데이터에서 각 항목의 평균 기여 규모 추정"""
        op_mask = df['Is_Operating_Flag'] == 2

        self.cost_mean = (df['Usage_kWh'] * current_price_avg).mean()
        self.epe_mean  = df.loc[op_mask, 'EPE'].mean() if op_mask.sum() > 0 else 1.0
        self.ocs_mean  = df['OCS'].mean()

        print(f"비용 평균:    {self.cost_mean:.1f} 원/스텝")
        print(f"EPE 평균:     {self.epe_mean:.4f}")
        print(f"OCS 평균:     {self.ocs_mean:.3f}")
        return self

    def suggest_lambdas(self, epe_ratio: float = 0.25, cont_ratio: float = 0.10):
        """
        비용 대비 보상 비율로 λ 자동 계산.

        Args:
            epe_ratio:  EPE 보상이 비용의 최대 몇 % 수준인지 (기본 25%)
            cont_ratio: OCS 보상이 비용의 최대 몇 % 수준인지 (기본 10%)

        Returns:
            (lambda_epe, lambda_cont)
        """
        assert self.cost_mean is not None, "fit()을 먼저 실행하세요."

        # λ = 목표 보상 규모 / 지표 평균
        lambda_epe  = (self.cost_mean * epe_ratio)  / (self.epe_mean  + 1e-8)
        lambda_cont = (self.cost_mean * cont_ratio)  / (self.ocs_mean  + 1e-8)

        print(f"\n권장 λ_epe:  {lambda_epe:.2f}  (EPE 보상 ≈ 비용의 {epe_ratio*100:.0f}%)")
        print(f"권장 λ_cont: {lambda_cont:.2f}  (OCS 보상 ≈ 비용의 {cont_ratio*100:.0f}%)")
        return lambda_epe, lambda_cont


# ── 수정된 목적함수 (HybridFastSimulator에 통합용) ─────────────
def make_objective_with_reward(
    model_usage,
    model_pf,
    X_row,
    m_idx, c_idx,
    # 기존 파라미터
    current_price, is_lag_period, target_pf,
    cum_kWh, cum_q,
    max_lagging, max_leading,
    # 기존 탐색 범위
    base_m, base_c, target_prod,
    # 현재 시점 PRI (ProductionProxyBuilder에서 계산)
    current_ppi: float,          # 현재 타임스텝의 PPI 값
    current_ocs: float,          # 현재 타임스텝의 OCS 값
    # 보상 가중치
    lambda_psi:  float = 5.0,    # PSI 패널티 가중치
    lambda_epe:  float = 300.0,  # EPE 보상 가중치 (RewardScaler 권장값 사용)
    lambda_cont: float = 50.0,   # OCS 보상 가중치
):
    """
    보상항이 통합된 목적함수 팩토리.

    기존 HybridFastSimulator._optimize_step() 내부의
    def objective(trial) 를 이 함수로 교체.

    [변경 전 → 변경 후 대응]
    cost += m_rate × 0.001    →  cost − λ_cont × OCS_reward (대체)
    (역할 동일: 모터 과가동 억제. 하지만 생산 중엔 오히려 보상)
    """
    def objective(trial):
        # ── 제어 변수 탐색 (기존과 동일) ───────────────────
        m_rate = trial.suggest_float(
            'm_rate', max(target_prod, base_m - 2.5), min(100.0, base_m + 2.5)
        )
        c_rate = trial.suggest_float(
            'c_rate', max(0.0, base_c - 5.0), min(100.0, base_c + 5.0)
        )

        X_temp = X_row.copy()
        X_temp[m_idx] = m_rate
        X_temp[c_idx] = c_rate

        # ── 서로게이트 예측 (기존과 동일) ──────────────────
        usage  = model_usage.predict(X_temp.reshape(1, -1))[0]
        pf_val = model_pf.predict(X_temp.reshape(1, -1))[0]
        pf_val = np.clip(pf_val, -1.0, 1.0)

        q_mag      = usage * np.sqrt(max(0, 1.0 / (pf_val**2 + 1e-9) - 1.0))
        q_lag_opt  = (m_rate / 100.0) * max_lagging
        q_lead_opt = (c_rate / 100.0) * max_leading
        is_lagging_state = (q_lag_opt - q_lead_opt) >= 0
        q_eval = q_mag if (is_lag_period and is_lagging_state)                        or (not is_lag_period and not is_lagging_state) else 0.0

        proj_cum_kWh = cum_kWh + usage
        proj_cum_q   = cum_q   + q_eval
        proj_app     = np.sqrt(proj_cum_kWh**2 + proj_cum_q**2)
        proj_pf      = proj_cum_kWh / proj_app if proj_app > 0 else 1.0

        # ── 비용 항 (기존과 동일) ───────────────────────────
        cost = usage * current_price

        # ── 역률 패널티/보상 (기존과 동일) ─────────────────
        if proj_pf < target_pf:
            cost += (target_pf - proj_pf) * 1_000_000.0
        if is_lag_period and proj_pf > target_pf:
            reward_pf_cap  = min(proj_pf, 0.95)
            reward_rate    = (reward_pf_cap - target_pf) * 100 * 0.002
            cost -= usage * current_price * reward_rate * 50

        # ── [기존] m_rate × 0.001 앵커 제거 ─────────────────────
        # OCS 보상으로 대체 (생산 중 모터 가동을 보상해야 하므로 단순 앵커 부적합)

        # ── [신규 ①] EPE 보상 ─────────────────────────────
        # CO2 신호 구조:
        #   CO2_observed = CO2_background + CO2_fuel + CO2_process_byproduct
        #   CO2_fuel, CO2_process_byproduct 모두 생산 활동에 비례
        #   → PPI 전체가 유효한 생산 프록시 신호
        #
        # CO2-Usage 단조 관계 약할 때 (Spearman r < 0.2):
        #   → lambda_epe 줄이고 lambda_cont 높이거나
        #   → ProductionProxyBuilder.validate() 의 lambda 조정 가이드 참고
        epe_reward = current_ppi / max(usage, 1.0)
        cost -= lambda_epe * epe_reward

        # ── [신규 ②] OCS 보상 (가동 연속성) ──────────────
        # m_rate 연동: 모터 가동 중일 때만 OCS 보상 활성화
        # 단순 앵커(m_rate × 0.001) 대비 물리적으로 의미 있는 대체
        ocs_this_step = current_ocs * (m_rate / 100.0)
        cost -= lambda_cont * ocs_this_step

        return cost

    return objective


print("✅ 보상항 통합 목적함수 (make_objective_with_reward) 정의 완료")


✅ 보상항 통합 목적함수 (make_objective_with_reward) 정의 완료


## 10.3 λ 가중치 민감도 분석 — 보상/패널티 균형 검증

In [12]:
def run_reward_lambda_sensitivity(
    df,
    model_usage,
    features,
    n_sample: int = 500,
    lambda_epe_candidates:  list = [0, 100, 300, 600, 1000],
    lambda_cont_candidates: list = [0, 20,  50,  100, 200],
):
    """
    λ_epe, λ_cont 조합별 목적함수 기여 분포 분석.

    목적:
        - 보상항이 패널티를 압도하거나 무시되지 않도록
          적절한 λ 범위 사전 탐색
        - 실제 Optuna 실행 전 스케일 검증

    출력:
        각 λ 조합에서의 보상항 / 비용항 비율 분포 히트맵
    """
    valid_features = [f for f in features if f in df.columns]
    sample_df = df.dropna(subset=valid_features + ['Usage_kWh', 'PPI', 'OCS']).sample(
        min(n_sample, len(df)), random_state=42
    )

    X_sample = sample_df[valid_features].values.astype(float)
    results = []

    for lam_epe in lambda_epe_candidates:
        for lam_cont in lambda_cont_candidates:
            epe_rewards  = []
            cont_rewards = []
            costs        = []

            for i, row in sample_df.iterrows():
                usage    = float(model_usage.predict(
                    X_sample[sample_df.index.get_loc(i)].reshape(1,-1))[0])
                ppi      = float(row.get('PPI', 0))
                ocs      = float(row.get('OCS', 0.5))
                m_rate   = float(row.get('Motor_Operating_Rate', 50))
                price    = 50.0  # 단순화된 단가

                cost        = usage * price
                epe_r       = lam_epe  * (ppi / max(usage, 1.0))
                cont_r      = lam_cont * ocs * (m_rate / 100.0)

                costs.append(cost)
                epe_rewards.append(epe_r)
                cont_rewards.append(cont_r)

            mean_cost  = np.mean(costs)
            results.append({
                'lambda_epe':  lam_epe,
                'lambda_cont': lam_cont,
                'epe_ratio':   np.mean(epe_rewards) / (mean_cost + 1e-8) * 100,
                'cont_ratio':  np.mean(cont_rewards) / (mean_cost + 1e-8) * 100,
                'total_reward_ratio': (np.mean(epe_rewards) + np.mean(cont_rewards))
                                      / (mean_cost + 1e-8) * 100
            })

    res_df = pd.DataFrame(results)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, col, title, note in [
        (axes[0], 'epe_ratio',          'EPE 보상 / 비용 (%)',
         '권장: 15~30%'),
        (axes[1], 'cont_ratio',         'OCS 보상 / 비용 (%)',
         '권장: 5~15%'),
        (axes[2], 'total_reward_ratio', '총 보상 / 비용 (%)',
         '권장: 20~40%'),
    ]:
        pivot = res_df.pivot(index='lambda_epe', columns='lambda_cont', values=col)
        sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax,
                    linewidths=0.5)
        ax.set_title(f'{title}\n{note}', fontsize=12, fontweight='bold')
        ax.set_xlabel('λ_cont')
        ax.set_ylabel('λ_epe')

    plt.suptitle('λ 가중치 민감도 분석\n(녹색 = 권장 범위, 적색 = 과도)', fontsize=14)
    plt.tight_layout()
    plt.show()

    # 권장 λ 조합 자동 선택 (총 보상 비율 20~35% 구간)
    optimal = res_df[
        (res_df['total_reward_ratio'] >= 20) &
        (res_df['total_reward_ratio'] <= 35)
    ]
    if len(optimal) > 0:
        best = optimal.iloc[0]
        print(f"\n✅ 권장 λ 조합:")
        print(f"   λ_epe  = {best['lambda_epe']:.0f}  "
              f"(EPE 보상 ≈ 비용의 {best['epe_ratio']:.1f}%)")
        print(f"   λ_cont = {best['lambda_cont']:.0f}  "
              f"(OCS 보상 ≈ 비용의 {best['cont_ratio']:.1f}%)")
    else:
        print("\n⚠️ 권장 범위의 λ 조합 없음. 후보 범위를 조정하세요.")

    return res_df


# ──── 실행 예시 ─────────────────────────────────────────────
# if 'df' in dir() and 'PPI' in df.columns and 'model_usage' in dir():
#     sensitivity_df = run_reward_lambda_sensitivity(df, model_usage, features)

print("✅ λ 민감도 분석 함수 정의 완료")


✅ λ 민감도 분석 함수 정의 완료


## 10.4 HybridFastSimulator 통합 패치 — 실행 가이드

In [13]:
"""
[HybridFastSimulator에 보상항 통합하는 방법]

기존 코드를 최소한으로 수정하는 2가지 방법 중 선택:

Method A (권장): _optimize_step() 내부 objective만 교체
    - 기존 run_scenario() 루프 구조 100% 보존
    - PPI/OCS 컬럼이 df에 있어야 함 (ProductionProxyBuilder.fit_transform 선행)

Method B: HybridFastSimulator 상속으로 확장
    - 기존 클래스 원본 수정 없음 (안전)
"""

class RewardAwareSimulator:
    """
    Method B: HybridFastSimulator를 래핑해 보상항 주입.

    사용:
        sim = RewardAwareSimulator(simulator, df_with_pri,
                                   lambda_epe=300, lambda_cont=50)
        result_df = sim.run_scenario_with_reward(df, scenario_key)

    내부적으로 기존 simulator.run_scenario()를 호출하되,
    각 타임스텝에서 목적함수에 보상항을 주입.
    """

    def __init__(
        self,
        base_simulator,          # 기존 HybridFastSimulator 인스턴스
        df_with_reward: pd.DataFrame,  # PPI, OCS, PRI 컬럼 포함된 df
        lambda_epe:  float = 300.0,
        lambda_cont: float = 50.0,
        lambda_psi:  float = 5.0,
    ):
        self.sim        = base_simulator
        self.reward_df  = df_with_reward.copy()
        self.lam_epe    = lambda_epe
        self.lam_cont   = lambda_cont
        self.lam_psi    = lambda_psi

    def get_reward_at(self, idx: int) -> dict:
        """특정 인덱스의 보상 관련 값 반환"""
        if idx < len(self.reward_df):
            row = self.reward_df.iloc[idx]
            return {
                'ppi': float(row.get('PPI', 0.0)),
                'ocs': float(row.get('OCS', 0.5)),
                'pri': float(row.get('PRI', 0.0)),
            }
        return {'ppi': 0.0, 'ocs': 0.5, 'pri': 0.0}

    def compute_reward_bonus(self, m_rate: float, usage: float,
                              reward_vals: dict) -> float:
        """
        보상 항 계산 (목적함수에서 차감할 값).

        Returns:
            reward_total: 클수록 이 제어값이 생산에 유리함
        """
        epe_reward  = self.lam_epe  * (reward_vals['ppi'] / max(usage, 1.0))
        cont_reward = self.lam_cont * reward_vals['ocs'] * (m_rate / 100.0)
        return epe_reward + cont_reward

    def summarize_reward_impact(self, result_df: pd.DataFrame):
        """
        최적화 결과에서 보상항이 미친 영향 분석.

        출력:
        - 보상 구간 vs 비보상 구간의 AI 절감액 비교
        - 가동 연속성 개선 여부
        - EPE 개선 전후 비교
        """
        if 'PRI' not in result_df.columns:
            print("PRI 컬럼 없음. ProductionProxyBuilder.transform()을 먼저 실행하세요.")
            return

        high_reward = result_df['PRI'] > result_df['PRI'].median()
        low_reward  = ~high_reward

        print("\n=== 보상항 영향 분석 ===")
        print(f"{'구간':<15} {'AI 절감 평균':>12} {'OCS 평균':>10} {'EPE 평균':>10}")
        print("-" * 50)
        for mask, label in [(high_reward, '고보상 구간'), (low_reward, '저보상 구간')]:
            sub = result_df[mask]
            saving = (sub['Usage_kWh'] - sub['AI_Usage_kWh']).mean() if 'AI_Usage_kWh' in sub else 0
            ocs    = sub['OCS'].mean()   if 'OCS' in sub else 0
            epe    = sub['EPE'].mean()   if 'EPE' in sub else 0
            print(f"{label:<15} {saving:>12.2f} kWh  {ocs:>10.3f}  {epe:>10.4f}")

        # 시각화
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        if 'AI_Usage_kWh' in result_df.columns:
            result_df['Saving_kWh'] = result_df['Usage_kWh'] - result_df['AI_Usage_kWh']
            axes[0].scatter(result_df['PRI'], result_df['Saving_kWh'],
                          alpha=0.3, s=10, color='dodgerblue')
            axes[0].axhline(0, color='red', ls='--')
            axes[0].set_title('PRI vs AI 절감량\n(우상단이 이상적)', fontsize=12)
            axes[0].set_xlabel('PRI (Production Reward Index)')
            axes[0].set_ylabel('절감량 (kWh)')
            axes[0].grid(True, alpha=0.3)

        axes[1].hist(result_df.loc[high_reward, 'OCS'], bins=30,
                    alpha=0.6, label='고보상 구간', color='green')
        axes[1].hist(result_df.loc[low_reward, 'OCS'], bins=30,
                    alpha=0.6, label='저보상 구간', color='gray')
        axes[1].set_title('가동 연속성(OCS) 분포\n(보상 구간 vs 비보상 구간)', fontsize=12)
        axes[1].set_xlabel('OCS')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()


print("✅ RewardAwareSimulator 정의 완료")
print()
print("📋 전체 실행 순서:")
print("  1. prod_builder = ProductionProxyBuilder()")
print("  2. df = prod_builder.fit_transform(df)         # PPI, EPE, OCS, PRI 생성")
print("  3. prod_builder.validate(df)                   # 유효성 확인")
print("  4. sensitivity_df = run_reward_lambda_sensitivity(df, model_usage, features)")
print("  5. reward_sim = RewardAwareSimulator(simulator, df, lambda_epe=300, lambda_cont=50)")
print("  6. reward_sim.summarize_reward_impact(result_df)  # 결과 분석")


✅ RewardAwareSimulator 정의 완료

📋 전체 실행 순서:
  1. prod_builder = ProductionProxyBuilder()
  2. df = prod_builder.fit_transform(df)         # PPI, EPE, OCS, PRI 생성
  3. prod_builder.validate(df)                   # 유효성 확인
  4. sensitivity_df = run_reward_lambda_sensitivity(df, model_usage, features)
  5. reward_sim = RewardAwareSimulator(simulator, df, lambda_epe=300, lambda_cont=50)
  6. reward_sim.summarize_reward_impact(result_df)  # 결과 분석


In [14]:
## 10.5 설계 정리: 목적함수 Before / After
"""
─────────────────────────────────────────────────────────────
[Before] 패널티 전용 구조
─────────────────────────────────────────────────────────────

J = usage × price                  ← 에너지 비용
  + m_rate × 0.001                 ← 앵커 (모터 억제, 퇴화 해 방지용)
  + PF_penalty × 1,000,000         ← 역률 위반
  - PF_reward                      ← 역률 개선 보상 (일부)

문제:
  ① m_rate × 0.001 → 모터 낮출수록 무조건 유리 (생산 무시)
  ② 공정 멈춤 = 비용 0 = 수학적 최적해 가능 (퇴화)
  ③ 에너지 절감 ≠ 에너지 효율 혼동

─────────────────────────────────────────────────────────────
[After] 보상-패널티 균형 구조
─────────────────────────────────────────────────────────────

J = usage × price                  ← 에너지 비용 (유지)
  + PF_penalty × 1,000,000         ← 역률 위반 (유지)
  - PF_reward                      ← 역률 개선 보상 (유지)
  - λ_epe  × EPE_reward            ← [신규] 단위 전력당 생산성
  - λ_cont × OCS_reward            ← [신규] 가동 연속성 (앵커 대체)

효과:
  ① 에너지 절감 + 생산성 동시 최적화 가능
  ② 공정 가동 중에는 모터 가동이 보상됨 → 퇴화 해 방지
  ③ OCS 보상이 m_rate × 0.001 앵커를 더 물리적으로 대체
  ④ CO2 데이터의 생산 정보가 최적화에 직접 반영됨

─────────────────────────────────────────────────────────────
[λ 설정 권장값 (RewardScaler 기반)]
─────────────────────────────────────────────────────────────

  λ_epe  ≈ 200~400  (EPE 보상 = 비용의 20~30%)
  λ_cont ≈  30~80   (OCS 보상 = 비용의  5~15%)
  → run_reward_lambda_sensitivity()로 데이터 기반 확인 권장
─────────────────────────────────────────────────────────────
"""
print("📌 Section 10 완료. 위 설명 참고하여 HybridFastSimulator objective에 통합하세요.")


📌 Section 10 완료. 위 설명 참고하여 HybridFastSimulator objective에 통합하세요.
